# Multimodal Sentiment Analysis and Topic Modelling of Bali Tourism Reviews

Pipeline:

1. **Compile dataset** – merge scraped Google Maps reviews into one table.
2. **Download images** – fetch the first photo attached to each review.
3. **Caption images** – describe each photo with BLIP.
4. **Clean dataset** – attach captions and drop rows missing text or caption.
5. **Sentiment classification** – compare text + caption, text only, caption only, and CLIP image embeddings.
6. **Topic modelling** – BERTopic on the review text.

Steps 1–4 produce `Dataset.csv`, `Dataset_with_captions.csv` and `Dataset_clean.csv`. If those files already exist you can start from step 5.

# Setup

In [ ]:
%pip install -q bertopic sentence-transformers umap-learn

In [ ]:
import copy
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Project folder: set the DATA_DIR environment variable to override.
DATA_DIR = Path(os.environ.get(
    "DATA_DIR",
    "/content/drive/MyDrive/Multimodal Bali" if IN_COLAB else ".",
))
RAW_DIR = DATA_DIR / "raw_datasets"
IMAGE_DIR = DATA_DIR / "images"
CAPTION_DIR = DATA_DIR / "captions"

DATASET_PATH = DATA_DIR / "Dataset.csv"
DATASET_CAPTIONS_PATH = DATA_DIR / "Dataset_with_captions.csv"
DATASET_CLEAN_PATH = DATA_DIR / "Dataset_clean.csv"
BERTOPIC_RESULT_PATH = DATA_DIR / "bertopic_result.csv"
BERTOPIC_TOPIC_INFO_PATH = DATA_DIR / "bertopic_topic_info.csv"

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("DATA_DIR:", DATA_DIR.resolve(), "| device:", device)

# 1. Compile Dataset

In [ ]:
csv_files = sorted(RAW_DIR.glob("*.csv"))
raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

selected_columns = ["placeId", "reviewId", "reviewImageUrls/0", "stars", "textTranslated", "title"]
df = (
    raw[selected_columns]
    .dropna(subset=["textTranslated"])
    .reset_index(drop=True)
)

df.to_csv(DATASET_PATH, index=False)
print(f"{len(csv_files)} raw files -> {df.shape[0]} reviews saved to {DATASET_PATH}")
df.head()

# 2. Download Images

In [ ]:
import requests
from tqdm.auto import tqdm

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATASET_PATH)

downloaded, skipped, failed = 0, 0, []
for review_id, url in tqdm(zip(df["reviewId"].astype(str), df["reviewImageUrls/0"]), total=len(df)):
    if pd.isna(url) or not str(url).strip():
        continue
    save_path = IMAGE_DIR / f"{review_id}.jpg"
    if save_path.exists():
        skipped += 1
        continue
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        save_path.write_bytes(response.content)
        downloaded += 1
    except requests.RequestException as e:
        failed.append((review_id, str(e)))

print(f"Downloaded: {downloaded} | already present: {skipped} | failed: {len(failed)}")

# 3. Caption Images (BLIP)

In [ ]:
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor

CAPTION_DIR.mkdir(parents=True, exist_ok=True)

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device).eval()

image_files = sorted(IMAGE_DIR.glob("*.jpg"))
errors = []
for img_path in tqdm(image_files):
    txt_path = CAPTION_DIR / f"{img_path.stem}.txt"
    if txt_path.exists():
        continue
    try:
        image = Image.open(img_path).convert("RGB")
        inputs = blip_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            output = blip_model.generate(**inputs, max_length=30)
        txt_path.write_text(blip_processor.decode(output[0], skip_special_tokens=True))
    except Exception as e:  # corrupt or non-image downloads
        errors.append((img_path.name, str(e)))

print(f"{len(image_files)} images, {len(errors)} caption errors")

# 4. Attach Captions and Remove Missing Values

In [ ]:
df = pd.read_csv(DATASET_PATH)


def load_caption(review_id):
    path = CAPTION_DIR / f"{review_id}.txt"
    return path.read_text().strip() if path.exists() else None


df["image_caption"] = df["reviewId"].astype(str).apply(load_caption)
df.to_csv(DATASET_CAPTIONS_PATH, index=False)

df_clean = df.dropna(subset=["textTranslated", "image_caption"]).reset_index(drop=True)
df_clean.to_csv(DATASET_CLEAN_PATH, index=False)

print(f"{len(df)} reviews -> {len(df_clean)} with both text and caption")
df_clean.info()

# 5. Sentiment Classification

Star ratings are mapped to three classes: **0 = negative** (1–2 stars), **1 = neutral** (3 stars), **2 = positive** (4–5 stars).
The positive class is downsampled to `N_MAJORITY` reviews so the classes are roughly balanced, then split 70 / 15 / 15 (stratified).
Every experiment uses the same balanced sample and the same split, so the results are directly comparable.

## Shared helpers

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

N_MAJORITY = 151
LABEL_NAMES = ["negative", "neutral", "positive"]


def star_to_label(star):
    if star <= 2:
        return 0
    if star == 3:
        return 1
    return 2


def load_balanced_dataset(n_majority=N_MAJORITY, seed=SEED):
    data = pd.read_csv(DATASET_CLEAN_PATH)
    data["label"] = data["stars"].apply(star_to_label)
    positive = data[data["label"] == 2].sample(n=n_majority, random_state=seed)
    balanced = pd.concat([data[data["label"] != 2], positive])
    return balanced.sample(frac=1, random_state=seed).reset_index(drop=True)


def split_indices(labels, seed=SEED):
    idx = np.arange(len(labels))
    train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=seed, stratify=labels)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=seed, stratify=np.asarray(labels)[temp_idx]
    )
    return train_idx, val_idx, test_idx


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            # deepcopy: state_dict() returns live references that keep changing during training
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience


def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, preds, labels = 0.0, [], []
    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            y = batch.pop("label")
            logits = model(**batch)
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            preds.extend(logits.argmax(dim=1).cpu().tolist())
            labels.extend(y.cpu().tolist())
    return total_loss / len(loader), labels, preds


def train_and_evaluate(name, model, datasets, lr, epochs, patience, batch_size=16):
    set_seed()
    train_ds, val_ds, test_ds = datasets
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(1, epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, _, _ = run_epoch(model, val_loader, criterion)
        print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")
        if early_stopping.step(val_loss, model):
            print(f"Early stopping at epoch {epoch}; best val loss {early_stopping.best_loss:.4f}")
            break

    model.load_state_dict(early_stopping.best_state)

    print(f"\n===== {name} =====")
    for split_name, loader in [("TRAIN", train_loader), ("VALIDATION", val_loader), ("TEST", test_loader)]:
        _, y_true, y_pred = run_epoch(model, loader, criterion)
        print(f"\n--- {split_name} --- accuracy {accuracy_score(y_true, y_pred):.4f}")
        print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, zero_division=0))
        print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

    return {
        "experiment": name,
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


df_sent = load_balanced_dataset()
train_idx, val_idx, test_idx = split_indices(df_sent["label"].values)
print("Class distribution after balancing:")
print(df_sent["label"].value_counts().sort_index())
print(f"Train / val / test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")

results = []

## BERT text classifier

In [ ]:
BERT_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)


class TextDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BERT_NAME)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        cls = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(cls))


def run_text_experiment(name, build_input):
    texts = df_sent.apply(build_input, axis=1).values
    labels = df_sent["label"].values
    datasets = [TextDataset(texts[i], labels[i]) for i in (train_idx, val_idx, test_idx)]
    set_seed()
    return train_and_evaluate(name, TextClassifier(), datasets, lr=2e-5, epochs=100, patience=5)

### Text + image caption

In [ ]:
results.append(run_text_experiment(
    "Text + caption",
    lambda r: f"<caption> {r['image_caption']} </caption> <text> {r['textTranslated']} </text>",
))

### Text only

In [ ]:
results.append(run_text_experiment(
    "Text only",
    lambda r: f"<text> {r['textTranslated']} </text>",
))

### Image caption only

In [ ]:
results.append(run_text_experiment(
    "Caption only",
    lambda r: f"<image_caption> {r['image_caption']} </image_caption>",
))

## CLIP image embeddings

Each review photo is encoded with CLIP (ViT-B/32) and L2-normalised; a small MLP is trained on top.
Reviews whose image file is missing or unreadable are reported and dropped from all three splits.

In [ ]:
from PIL import Image
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)


def image_embedding(review_id):
    path = IMAGE_DIR / f"{review_id}.jpg"
    try:
        image = Image.open(path).convert("RGB")
    except (FileNotFoundError, OSError):
        return None
    inputs = clip_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs).squeeze(0)
    return (emb / emb.norm(p=2).clamp_min(1e-8)).cpu()


embeddings = [image_embedding(rid) for rid in tqdm(df_sent["reviewId"].astype(str))]
has_image = np.array([e is not None for e in embeddings])
print(f"Images found for {has_image.sum()} / {len(has_image)} reviews")


class EmbeddingDataset(Dataset):
    def __init__(self, idx):
        idx = [i for i in idx if has_image[i]]
        self.embeddings = [embeddings[i] for i in idx]
        self.labels = df_sent["label"].values[idx]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"image": self.embeddings[i], "label": torch.tensor(self.labels[i], dtype=torch.long)}


class ImageClassifier(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 3),
        )

    def forward(self, image):
        return self.classifier(image)


datasets = [EmbeddingDataset(i) for i in (train_idx, val_idx, test_idx)]
set_seed()
# The MLP is trained from scratch, so it needs a much higher learning rate than BERT fine-tuning.
results.append(train_and_evaluate(
    "CLIP image embedding", ImageClassifier(), datasets, lr=1e-3, epochs=200, patience=10,
))

## Comparison

In [ ]:
pd.DataFrame(results).set_index("experiment").round(4)

# 6. Topic Modelling (BERTopic)

Sentence embeddings are computed from the original review text. The stop-word-free `clean_text` is only used
for the c-TF-IDF topic representations, so stop-word removal doesn't degrade the embeddings.
UMAP is seeded so topics are reproducible.

In [ ]:
import re
import string

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score
from umap import UMAP

df = pd.read_csv(DATASET_CLEAN_PATH).dropna(subset=["textTranslated"]).reset_index(drop=True)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(w for w in text.split() if w not in ENGLISH_STOP_WORDS)


df["clean_text"] = df["textTranslated"].apply(clean_text)
docs = df["clean_text"].tolist()

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(df["textTranslated"].astype(str).tolist(), show_progress_bar=True)


def make_topic_model(nr_topics, calculate_probabilities=False, verbose=False):
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=SEED)
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        nr_topics=nr_topics,
        calculate_probabilities=calculate_probabilities,
        verbose=verbose,
    )

## Choose the number of topics

In [ ]:
topic_range = list(range(10, 25, 2))  # 10, 12, ..., 24
search = []
for k in topic_range:
    topics_k, _ = make_topic_model(k).fit_transform(docs, embeddings)
    topics_k = np.array(topics_k)
    mask = topics_k != -1  # ignore outliers
    score = (
        silhouette_score(embeddings[mask], topics_k[mask], metric="cosine")
        if len(set(topics_k[mask])) > 1 else -1
    )
    search.append((k, len(set(topics_k)), score))
    print(f"nr_topics={k:2d} | topics found={len(set(topics_k)):2d} | silhouette={score:.4f}")

results_df = pd.DataFrame(search, columns=["nr_topics", "num_topics", "silhouette"])
best_k = int(results_df.loc[results_df["silhouette"].idxmax(), "nr_topics"])
print("\nBest nr_topics:", best_k)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.figure(figsize=(10, 6))
sns.lineplot(data=results_df, x="nr_topics", y="silhouette", marker="o",
             linewidth=2.5, color="royalblue", label="Silhouette score")
best = results_df.loc[results_df["silhouette"].idxmax()]
plt.scatter(best["nr_topics"], best["silhouette"], color="red", s=120, label="Best score", zorder=3)
plt.title("BERTopic: Silhouette Score by Number of Topics", fontsize=16, weight="bold")
plt.xlabel("Number of topics (nr_topics)", fontsize=13)
plt.ylabel("Silhouette score", fontsize=13)
plt.xticks(results_df["nr_topics"])
plt.legend()
plt.tight_layout()
plt.show()

## Final model

In [ ]:
final_model = make_topic_model(best_k, calculate_probabilities=True, verbose=True)
topics, probs = final_model.fit_transform(docs, embeddings)
df["topic"] = topics
df["probs"] = list(probs)

topic_info = final_model.get_topic_info().sort_values("Topic").reset_index(drop=True)
for _, row in topic_info.iterrows():
    print(f"Topic {row['Topic']}: {row['Representation']}")
    for doc in row["Representative_Docs"]:
        print("   -", doc[:200])
    print()

In [ ]:
mask = df["topic"] != -1
df_plot = df[mask].copy()
umap_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine",
               random_state=SEED).fit_transform(embeddings[mask.values])
df_plot["x"], df_plot["y"] = umap_2d[:, 0], umap_2d[:, 1]

sns.set_theme(style="white", context="paper")
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_plot, x="x", y="y", hue="topic", palette="tab20", s=18, alpha=0.75, linewidth=0)
plt.title("BERTopic Clusters in 2D UMAP Space (Outliers Removed)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title="Topic", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv(BERTOPIC_RESULT_PATH, index=False)
topic_info.to_csv(BERTOPIC_TOPIC_INFO_PATH, index=False)
print("Saved:", BERTOPIC_RESULT_PATH, "and", BERTOPIC_TOPIC_INFO_PATH)

## Topic interpretation

> **Note:** these labels were assigned to the topic IDs from an earlier run. Topic IDs change whenever the
> model is refit, so re-check this mapping against the printed topics above after rerunning.

| Original topics | Merged theme |
|---|---|
| 1, 16 | Wildlife Encounters, Visitor Safety, and Environmental Conditions |
| 2, 8, 12 | Cultural and Natural Heritage Tourism Experience in Bali |
| 3, 13, 14, 15, 19 | General Tourist Satisfaction and Emotional Experience |
| 4 | Environmental Quality, Cleanliness, and Comfort |
| 5 | Traditional Balinese Village Tourism Experience |
| 6, 17 | Sunset and Temporal Scenic Aesthetic Experience |
| 7 | Cultural Performance Tourism: Kecak Dance Experience |
| 9 | Tourism Infrastructure, Accessibility, and Visitor Facilities |
| 10 | Generic Positive Sentiment Expressions |
| 11 | Aesthetic Awe and Strong Emotional Reaction |
| 18 | Family-Friendly and Educational Tourism Experience |
| 20 | Coastal Recreation and Surfing Tourism in Uluwatu |

- **Wildlife / safety** – interactions with animals (e.g. monkeys), visitor safety and site conditions.
- **Cultural & natural heritage** – temples, lakes, beaches, mountains and cultural landscapes, plus atmosphere and food services.
- **General satisfaction** – positive sentiment and enjoyment without a specific location or feature.
- **Environmental quality** – cleanliness, comfort, air quality and overall site aesthetics.
- **Traditional village** – traditional Balinese villages, cultural housing and traditional clothing.
- **Sunset** – sunset views, time-of-day atmosphere and scenic appreciation.
- **Kecak dance** – cultural performances, especially the Kecak dance at Uluwatu Temple.
- **Infrastructure** – parking, entrance fees, ticketing and facility management.
- **Generic positive** – very short positive expressions with little semantic content.
- **Aesthetic awe** – strong reactions such as "wow", "amazing", "speechless".
- **Family-friendly** – suitability for families and children, educational or recreational value.
- **Surfing** – surfing, waves, tides and beach recreation around Uluwatu.

## Inspect reviews

In [ ]:
def show_review(review_id):
    row = df.loc[df["reviewId"] == review_id].iloc[0]
    print(f"stars={row['stars']} | topic={row['topic']} | place={row['title']}\n{row['textTranslated']}")


def reviews_with_keyword(keyword):
    return df.loc[df["clean_text"].str.contains(keyword, case=False, na=False), ["reviewId", "topic", "textTranslated"]]


def reviews_in_topic(topic_id):
    return df.loc[df["topic"] == topic_id, ["reviewId", "stars", "textTranslated"]]


reviews_with_keyword("surf").head()